# SemanticDraw SD1.5 + LCM trên COCO Val2017 Semantic-Full

Notebook Colab này chạy **pipeline gốc của SemanticDraw SD1.5** với sampler **LCM**. Notebook này không dùng manifest 1073 mẫu đã lọc theo chuẩn `multidiffusion_coco_all`; thay vào đó nó có thể tự tạo manifest rộng hơn cho COCO val2017 theo profile `semanticdraw_sd15`.

Code lõi baseline được dùng trực tiếp:

```text
Baseline/semantic-draw-main/src/model/pipeline_semantic_draw.py
```

## Chế độ mặc định A100 80GB

Mặc định notebook chạy full COCO val2017 trên GPU mạnh như A100 80GB:

```python
RUN_PROFILE = "full_val2017"
LOW_VRAM = False
REBUILD_MANIFEST = False
RUN_SANITY_CHECK = True
RUN_METRICS = False
RUN_EXPORT_ZIP = True
SKIP_EXISTING = True

# Các giá trị bên dưới sẽ được notebook tự suy ra khi LOW_VRAM = False:
MAX_OBJECTS_PER_IMAGE = 80
BATCH_SIZE = 8
METRIC_BATCH_SIZE = 8
CLIP_BATCH_SIZE = 16
MAX_DISPLAY_RESULTS = 4
```

`BATCH_SIZE` trong notebook này là batch của dataloader. Phần generation vẫn gọi pipeline baseline theo từng sample, vì `SemanticDrawPipeline` gốc không batch hóa nhiều ảnh hoàn chỉnh theo kiểu một lần gọi sinh nhiều ảnh khác nhau.

## Nếu muốn validate nhanh bằng smoke test

Nếu chỉ muốn chạy nhẹ trên T4/L4 hoặc kiểm tra pipeline trước, đổi lại:

```python
RUN_PROFILE = "smoke"
LOW_VRAM = True
```

## Protocol đang dùng

```text
Model      : runwayml/stable-diffusion-v1-5
Sampler    : LCMScheduler được load bên trong SemanticDrawPipeline
Accel      : latent-consistency/lcm-lora-sdv1-5 do baseline SemanticDraw load
Resolution : 512x512
Data       : COCO val2017 captions + COCO instance masks
Profile    : semanticdraw_sd15, không phải multidiffusion_coco_all
Input API  : prompts = [caption] + foreground prompts
             masks   = [background mask] + foreground masks
```

Lệnh generation được căn cho giống notebook SemanticDraw SD1.5 + LCM 1073 trước đó: caption COCO được đưa vào như một vùng background rõ ràng, còn background mask được tính bằng `1 - union(foreground_masks)`.

Ở profile full, notebook giữ mọi ảnh COCO val2017 có ít nhất một object segmentation hợp lệ và không phải crowd. Notebook không lọc theo person category, không lọc theo tỷ lệ diện tích object, và không ép số object phải nằm trong khoảng 2-4. Khi `LOW_VRAM=True`, số mask mỗi ảnh bị giới hạn để validate nhanh. Khi `LOW_VRAM=False`, `FULL_MAX_OBJECTS_PER_IMAGE=80`, đủ bao phủ mức tối đa hiện tại của COCO val2017 là 62 mask object hợp lệ trong một ảnh.


In [ ]:
# 0. Cài thư viện cần thiết.
# Không cài lại torch trong Colab. Colab đã có sẵn bản torch tương thích CUDA.
import sys
import subprocess

packages = [
    "diffusers>=0.30.0",
    "transformers>=4.44.0",
    "accelerate",
    "peft",
    "huggingface_hub",
    "safetensors",
    "sentencepiece",
    "protobuf",
    "einops",
    "pycocotools",
    "matplotlib",
    "tqdm",
    "pandas>=2.0",
    "open-clip-torch>=2.24.0",
    "torch-fidelity>=0.3.0",
    "torchmetrics>=1.4",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
print("[OK] Đã cài thư viện. Đã gỡ torchao để tránh lỗi PEFT khi nạp LoRA.")


In [ ]:
# 1. Clone hoặc tìm repo AnchorDraw trong runtime.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/GOx9-P/AnchorDraw.git"
WORK_DIR = Path("/content")


def is_repo_root(path: Path) -> bool:
    return (
        (path / "Baseline" / "semantic-draw-main" / "src" / "model" / "pipeline_semantic_draw.py").exists()
        and (path / "Ours" / "src" / "data").exists()
    )


def find_repo_root() -> Path | None:
    starts = [
        Path.cwd(),
        Path.cwd() / "AnchorDraw",
        WORK_DIR / "AnchorDraw",
        WORK_DIR / "AnchorDraw" / "AnchorDraw",
    ]
    checked = set()
    for start in starts:
        if not start.exists():
            continue
        for path in [start, *start.parents]:
            path = path.resolve()
            if path in checked:
                continue
            checked.add(path)
            if is_repo_root(path):
                return path
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    clone_target = WORK_DIR / "AnchorDraw"
    if not clone_target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_target)], check=True)
    REPO_ROOT = find_repo_root()

assert REPO_ROOT is not None and is_repo_root(REPO_ROOT), "Không tìm thấy repo root sau khi clone."
print(f"[OK] Repo root: {REPO_ROOT}")


In [ ]:
# 2. Cấu hình thí nghiệm.
# Mặc định chạy full trên GPU mạnh như A100 80GB. Nếu chỉ validate nhanh thì đổi về RUN_PROFILE="smoke" và LOW_VRAM=True.
from pathlib import Path
import json

RUN_PROFILE = "full_val2017" # "smoke" hoặc "full_val2017"
LOW_VRAM = False             # False cho GPU mạnh như A100 80GB; True cho T4/L4 hoặc validate nhẹ
REBUILD_MANIFEST = False      # Đặt True nếu vừa đổi rule sampling, ví dụ MAX_OBJECTS_PER_IMAGE.
RUN_SANITY_CHECK = True       # Chạy một ảnh test nhanh để kiểm tra model/scheduler/VAE.
RUN_METRICS = True          # Metric nội bộ; protocol CLIP hiện tại chưa hoàn toàn khớp paper.
RUN_EXPORT_ZIP = True         # Tạo zip chứa ảnh sinh + manifest export để tải về.
SKIP_EXISTING = True          # Nếu ảnh đã tồn tại thì dùng lại, hữu ích khi resume.

COCO_ROOT = Path(os.environ.get("COCO_ROOT", "/content/COCO"))
BASE_OUTPUT_DIR = Path("/content/anchordraw_runs")

MODEL_ID = "runwayml/stable-diffusion-v1-5"
TARGET_SIZE = (512, 512)
BASE_SEED = 2024
MANIFEST_SEED = 2026
SMOKE_SUBSET_SIZE = 8
SMOKE_MAX_OBJECTS_PER_IMAGE = 8
FULL_MAX_OBJECTS_PER_IMAGE = 80  # COCO val2017 hiện có tối đa 62 mask object hợp lệ trong một ảnh.
MAX_OBJECTS_PER_IMAGE = SMOKE_MAX_OBJECTS_PER_IMAGE if (RUN_PROFILE == "smoke" or LOW_VRAM) else FULL_MAX_OBJECTS_PER_IMAGE
DROP_ISCROWD = True

BATCH_SIZE = 1 if LOW_VRAM else 8
METRIC_BATCH_SIZE = 1 if LOW_VRAM else 8
CLIP_BATCH_SIZE = 4 if LOW_VRAM else 16
NUM_WORKERS = 0

BOOTSTRAP_STEPS = 1
MASK_STD = 0.0
MASK_STRENGTH = 1.0
PREPROCESS_MASK_COVER_ALPHA = 0.0
MASK_TYPE = "discrete"
NEGATIVE_PROMPT = ""
NUM_INFERENCE_STEPS = None   # None nghĩa là dùng default của baseline SemanticDrawPipeline.
GUIDANCE_SCALE = None        # None nghĩa là dùng default của baseline SemanticDrawPipeline.

START_INDEX = 0
MAX_SAMPLES = None           # None nghĩa là chạy hết range đã chọn.
MAX_DISPLAY_RESULTS = 8 if RUN_PROFILE == "smoke" else 4

# Preset tham chiếu đang khớp với mặc định để chạy full trên A100 80GB.
# Nếu muốn smoke test nhanh, hãy đặt RUN_PROFILE="smoke" và LOW_VRAM=True ở phía trên.
A100_80GB_FULL_PRESET = {
    "RUN_PROFILE": "full_val2017",
    "LOW_VRAM": False,
    "REBUILD_MANIFEST": False,
    "RUN_SANITY_CHECK": True,
    "RUN_METRICS": False,
    "RUN_EXPORT_ZIP": True,
    "SKIP_EXISTING": True,
    "MAX_OBJECTS_PER_IMAGE": FULL_MAX_OBJECTS_PER_IMAGE,
    "BATCH_SIZE": 8,
    "METRIC_BATCH_SIZE": 8,
    "CLIP_BATCH_SIZE": 16,
    "BOOTSTRAP_STEPS": BOOTSTRAP_STEPS,
    "MASK_STD": MASK_STD,
    "MASK_STRENGTH": MASK_STRENGTH,
    "NUM_INFERENCE_STEPS": NUM_INFERENCE_STEPS,
    "GUIDANCE_SCALE": GUIDANCE_SCALE,
    "START_INDEX": 0,
    "MAX_SAMPLES": None,
    "MAX_DISPLAY_RESULTS": 4,
    "ghi_chu": "BATCH_SIZE là batch của dataloader; pipeline baseline vẫn sinh từng sample một.",
}

if RUN_PROFILE == "smoke":
    subset_size = SMOKE_SUBSET_SIZE
elif RUN_PROFILE == "full_val2017":
    subset_size = None
else:
    raise ValueError(f"RUN_PROFILE không được hỗ trợ: {RUN_PROFILE}")

mode_name = "low_vram" if LOW_VRAM else "standard"
run_suffix = "smoke" if RUN_PROFILE == "smoke" else "full_val2017"
RUN_ID = f"semanticdraw_sd15_lcm_semanticfull_512x512_{run_suffix}_{mode_name}"
RUN_ROOT = BASE_OUTPUT_DIR / RUN_ID
GENERATED_IMAGES_DIR = RUN_ROOT / "generated_images"
OVERLAY_DIR = RUN_ROOT / "mask_overlays"
MANIFEST_DIR = RUN_ROOT / "manifests"
MASK_CACHE_DIR = RUN_ROOT / "mask_cache"
METRICS_OUTPUT_DIR = RUN_ROOT / "metrics"
RUN_SUMMARY_PATH = RUN_ROOT / "generation_summary.json"
RUN_CONFIG_PATH = RUN_ROOT / "run_config.json"

MANIFEST_NAME = (
    f"coco_val2017_semanticdraw_sd15_512x512_maxobj{MAX_OBJECTS_PER_IMAGE}_seed{MANIFEST_SEED}_{subset_size}.jsonl"
    if subset_size is not None
    else f"coco_val2017_semanticdraw_sd15_512x512_maxobj{MAX_OBJECTS_PER_IMAGE}_full_val2017.jsonl"
)
RUN_MANIFEST = MANIFEST_DIR / MANIFEST_NAME

for path in [RUN_ROOT, GENERATED_IMAGES_DIR, OVERLAY_DIR, MANIFEST_DIR, MASK_CACHE_DIR, METRICS_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

run_config = {
    "run_id": RUN_ID,
    "run_profile": RUN_PROFILE,
    "low_vram": LOW_VRAM,
    "model_id": MODEL_ID,
    "sampler": "LCMScheduler qua baseline SemanticDrawPipeline",
    "resolution": list(TARGET_SIZE),
    "manifest_path": str(RUN_MANIFEST),
    "coco_root": str(COCO_ROOT),
    "subset_size": subset_size,
    "max_objects_per_image": MAX_OBJECTS_PER_IMAGE,
    "smoke_max_objects_per_image": SMOKE_MAX_OBJECTS_PER_IMAGE,
    "full_max_objects_per_image": FULL_MAX_OBJECTS_PER_IMAGE,
    "drop_iscrowd": DROP_ISCROWD,
    "batch_size": BATCH_SIZE,
    "metric_batch_size": METRIC_BATCH_SIZE,
    "clip_batch_size": CLIP_BATCH_SIZE,
    "bootstrap_steps": BOOTSTRAP_STEPS,
    "mask_std": MASK_STD,
    "mask_strength": MASK_STRENGTH,
    "mask_type": MASK_TYPE,
    "start_index": START_INDEX,
    "max_samples": MAX_SAMPLES,
    "a100_80gb_full_preset": A100_80GB_FULL_PRESET,
}
with RUN_CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

print("[CẤU HÌNH ĐANG CHẠY]")
print(json.dumps(run_config, ensure_ascii=False, indent=2))
print("\n[PRESET FULL CHO A100 80GB]")
print(json.dumps(A100_80GB_FULL_PRESET, ensure_ascii=False, indent=2))

if RUN_PROFILE == "full_val2017":
    print("[WARN] Output full được lưu trong runtime Colab. Hãy tải file zip export trước khi kết thúc session.")
    if MAX_OBJECTS_PER_IMAGE < FULL_MAX_OBJECTS_PER_IMAGE:
        print("[WARN] LOW_VRAM đang giới hạn số mask mỗi ảnh. Dùng LOW_VRAM=False nếu muốn chạy full object-mask.")


In [ ]:
# 3. Tải COCO val2017 nếu runtime chưa có dữ liệu.
import ssl
import urllib.request
import zipfile

COCO_ROOT.mkdir(parents=True, exist_ok=True)

VAL_ZIP_URLS = [
    "http://images.cocodataset.org/zips/val2017.zip",
    "https://images.cocodataset.org/zips/val2017.zip",
]
ANN_ZIP_URLS = [
    "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    "https://images.cocodataset.org/annotations/annotations_trainval2017.zip",
]
val_zip = COCO_ROOT / "val2017.zip"
ann_zip = COCO_ROOT / "annotations_trainval2017.zip"


def run_download_command(cmd: list[str]) -> bool:
    try:
        subprocess.run(cmd, check=True)
        return True
    except Exception as exc:
        print(f"[WARN] Lệnh tải thất bại: {' '.join(cmd[:2])} -> {exc}")
        return False


def download_file(urls: list[str], dst: Path) -> None:
    if dst.exists() and dst.stat().st_size > 0:
        print(f"[BỎ QUA] Đã tải: {dst.name}")
        return
    last_error = None
    for url in urls:
        print(f"[TẢI] {url}")
        if run_download_command(["wget", "-c", "--no-check-certificate", "-O", str(dst), url]):
            if dst.exists() and dst.stat().st_size > 0:
                return
        if run_download_command(["curl", "-L", "-k", "--retry", "3", "-o", str(dst), url]):
            if dst.exists() and dst.stat().st_size > 0:
                return
        try:
            context = ssl._create_unverified_context()
            with urllib.request.urlopen(url, context=context, timeout=120) as response:
                with dst.open("wb") as f:
                    f.write(response.read())
            if dst.exists() and dst.stat().st_size > 0:
                return
        except Exception as exc:
            last_error = exc
            print(f"[WARN] urllib tải thất bại với {url}: {exc}")
    raise RuntimeError(f"Không tải được {dst.name}. Lỗi cuối: {last_error}")


def unzip_if_missing(zip_path: Path, marker_path: Path) -> None:
    if marker_path.exists():
        print(f"[BỎ QUA] Đã giải nén: {marker_path}")
        return
    print(f"[GIẢI NÉN] {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(COCO_ROOT)


download_file(VAL_ZIP_URLS, val_zip)
download_file(ANN_ZIP_URLS, ann_zip)
unzip_if_missing(val_zip, COCO_ROOT / "val2017" / "000000000139.jpg")
unzip_if_missing(ann_zip, COCO_ROOT / "annotations" / "instances_val2017.json")

assert (COCO_ROOT / "val2017").exists(), "Thiếu ảnh COCO val2017."
assert (COCO_ROOT / "annotations" / "instances_val2017.json").exists(), "Thiếu instances_val2017.json."
assert (COCO_ROOT / "annotations" / "captions_val2017.json").exists(), "Thiếu captions_val2017.json."
print("[OK] COCO val2017 đã sẵn sàng.")


In [ ]:
# 4. Import dataloader của Ours và pipeline SemanticDraw baseline gốc.
import sys
import importlib.util
import time
import math
import csv
import shutil
import gc

import torch
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown

OURS_SRC = REPO_ROOT / "Ours" / "src"
BASELINE_SRC = REPO_ROOT / "Baseline" / "semantic-draw-main" / "src"

sys.path.insert(0, str(OURS_SRC))
from data import semanticdraw_sd15, build_coco_manifest, build_coco_region_dataloader, batch_item_to_semanticdraw_inputs
from data.visualize import make_mask_overlay

sys.path.insert(0, str(BASELINE_SRC))
pipeline_path = BASELINE_SRC / "model" / "pipeline_semantic_draw.py"
spec = importlib.util.spec_from_file_location("pipeline_semantic_draw_original", pipeline_path)
pipeline_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(pipeline_module)
SemanticDrawPipeline = pipeline_module.SemanticDrawPipeline

print("[OK] Import xong.")
print(f"[OK] File baseline: {pipeline_path}")


In [ ]:
# 5. Tạo manifest SemanticDraw-full và khởi tạo dataloader.
# Cell này dùng profile SemanticDraw, không dùng filter MultiDiffusion COCO-all chặt hơn.
base_config = semanticdraw_sd15(
    COCO_ROOT,
    seed=MANIFEST_SEED,
    subset_size=subset_size,
    manifest_path=RUN_MANIFEST,
)

config = base_config.copy_with(
    instances_json=COCO_ROOT / "annotations" / "instances_val2017.json",
    captions_json=COCO_ROOT / "annotations" / "captions_val2017.json",
    min_objects=1,
    max_objects=MAX_OBJECTS_PER_IMAGE,
    truncate_objects=True,
    exclude_categories=(),
    min_mask_area_ratio=0.0,
    drop_iscrowd=DROP_ISCROWD,
    prompt_template="a {label}",
    caption_policy="first",
    object_policy="largest",
    return_image=True,
    cache_resized_masks=True,
    cache_dir=MASK_CACHE_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    persistent_workers=False,
)

records = build_coco_manifest(config, overwrite=REBUILD_MANIFEST)
loader = build_coco_region_dataloader(config, shuffle=False, drop_last=False)
dataset_size = len(loader.dataset)
num_batches = len(loader)
preview_batch = next(iter(loader))

print(f"[OK] Đường dẫn manifest: {RUN_MANIFEST}")
print(f"[OK] Số record trong manifest: {len(records)}")
print(f"[OK] Số record trong dataloader: {dataset_size}")
print(f"[OK] Số batch dataloader: {num_batches} x tối đa {BATCH_SIZE}")
print(f"[OK] Shape tensor mask batch đầu: {tuple(preview_batch['masks'].shape)}")
print("Các sample_id đầu tiên:")
for sample_id in preview_batch["sample_ids"][:8]:
    print(" -", sample_id)


In [ ]:
# 6. Các hàm hỗ trợ: chuẩn bị input baseline, hiển thị, resume và kiểm tra ảnh.
def md_escape(text: object) -> str:
    return str(text).replace("\n", " ").replace("|", "\\|")


def seed_everything(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def image_stats(image: Image.Image) -> dict:
    import numpy as np
    arr = np.asarray(image.convert("RGB"), dtype=np.uint8)
    return {
        "min": int(arr.min()),
        "max": int(arr.max()),
        "mean": float(arr.mean()),
        "std": float(arr.std()),
    }


def make_semanticdraw_payload(batch: dict, index: int) -> dict:
    item = batch_item_to_semanticdraw_inputs(batch, index)
    metadata = item["metadata"]

    # Khớp với notebook SemanticDraw SD1.5 + LCM 1073 trước đó:
    # caption COCO được đưa vào như một vùng background rõ ràng.
    fg_masks = item["masks"].float().cpu()  # (P, 1, H, W)
    fg_union = fg_masks.sum(dim=0, keepdim=True).clamp(0, 1)
    background_mask = (1.0 - fg_union).clamp(0, 1)
    all_masks = torch.cat([background_mask, fg_masks], dim=0)

    prompts = [item["background_prompt"], *item["prompts"]]
    negative_prompts = [NEGATIVE_PROMPT for _ in prompts]

    return {
        "sample_id": metadata["sample_id"],
        "image_id": metadata["image_id"],
        "file_name": metadata["file_name"],
        "height": item["height"],
        "width": item["width"],
        "background_prompt": item["background_prompt"],
        "prompts": prompts,
        "negative_prompts": negative_prompts,
        "foreground_prompts": item["prompts"],
        "category_names": metadata["category_names"],
        "annotation_ids": metadata["annotation_ids"],
        "area_ratios": metadata["area_ratios"],
        "foreground_masks": fg_masks,
        "all_masks": all_masks,
        "metadata": metadata,
    }


def generated_path_for(index: int, sample_id: str) -> Path:
    return GENERATED_IMAGES_DIR / f"{index:06d}_{sample_id}_generated.png"


def overlay_path_for(index: int, sample_id: str) -> Path:
    return OVERLAY_DIR / f"{index:06d}_{sample_id}_overlay.png"


def display_result(payload: dict, original: Image.Image, overlay: Image.Image, generated: Image.Image, elapsed, generated_path: Path) -> None:
    rows = ["| Vùng | Prompt | Annotation | Tỷ lệ diện tích |", "|---|---|---:|---:|"]
    rows.append(f"| Background | {md_escape(payload['background_prompt'])} | - | - |")
    for label, prompt, ann_id, area in zip(payload["category_names"], payload["foreground_prompts"], payload["annotation_ids"], payload["area_ratios"]):
        rows.append(f"| {md_escape(label)} | {md_escape(prompt)} | {ann_id} | {float(area):.4f} |")

    elapsed_text = "dùng lại ảnh cũ" if elapsed is None else f"{elapsed:.2f}s"
    display(Markdown(
        f"### `{payload['sample_id']}`\n"
        f"- image_id: `{payload['image_id']}`\n"
        f"- file: `{payload['file_name']}`\n"
        f"- số prompt/mask: `{len(payload['prompts'])}`\n"
        f"- đường dẫn ảnh sinh: `{generated_path}`\n"
        f"- thời gian: `{elapsed_text}`\n"
        f"- thống kê ảnh sinh: `{image_stats(generated)}`\n\n"
        + "\n".join(rows)
    ))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(original)
    axes[0].set_title("Ảnh COCO gốc đã resize")
    axes[1].imshow(overlay)
    axes[1].set_title("Overlay foreground mask")
    axes[2].imshow(generated)
    axes[2].set_title("Ảnh sinh bởi SemanticDraw SD1.5 LCM")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def load_existing_summary() -> dict[int, dict]:
    if not RUN_SUMMARY_PATH.exists():
        return {}
    with RUN_SUMMARY_PATH.open("r", encoding="utf-8") as f:
        rows = json.load(f)
    out = {}
    for row in rows:
        if "index" in row:
            out[int(row["index"])] = row
    return out


print("[OK] Các hàm hỗ trợ đã sẵn sàng.")


In [ ]:
# 7. Đăng nhập Hugging Face nếu có token, sau đó load pipeline baseline.
def maybe_login_to_huggingface() -> None:
    token = os.environ.get("HF_TOKEN")
    if token is None:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if token:
        from huggingface_hub import login
        login(token=token)
        print("[OK] Đã nạp Hugging Face token.")
    else:
        print("[INFO] Không thấy HF_TOKEN. Notebook sẽ dùng quyền truy cập public.")


assert torch.cuda.is_available(), "Colab chưa bật GPU. Vào Runtime > Change runtime type > GPU."
device = torch.device("cuda:0")
dtype = torch.float16
maybe_login_to_huggingface()
print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")

# Chặn lỗi tương thích torchao/PEFT trước khi nạp LCM LoRA.
import importlib
import importlib.util
importlib.invalidate_caches()
if "torchao" in sys.modules or importlib.util.find_spec("torchao") is not None:
    raise RuntimeError("torchao vẫn còn import được. Hãy restart runtime, chạy lại cell cài thư viện, rồi Run all.")

seed_everything(BASE_SEED)
smd = SemanticDrawPipeline(
    device=device,
    dtype=dtype,
    sd_version="1.5",
    hf_key=MODEL_ID,
    has_i2t=False,
    default_mask_std=MASK_STD,
    default_mask_strength=MASK_STRENGTH,
    default_preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA,
    mask_type=MASK_TYPE,
)

if hasattr(smd.pipe, "enable_attention_slicing"):
    smd.pipe.enable_attention_slicing()
if LOW_VRAM and hasattr(smd.pipe, "enable_vae_slicing"):
    smd.pipe.enable_vae_slicing()

print("[OK] SemanticDrawPipeline đã sẵn sàng.")


In [ ]:
# 8. Kiểm tra notebook đang dùng đúng baseline SD1.5 + LCM.
print("Họ model: SD1.5")
print("Model id:", MODEL_ID)
print("Scheduler:", type(smd.scheduler).__name__)
print("Scheduler trong pipeline:", type(smd.pipe.scheduler).__name__)
print("Số bước inference mặc định:", smd.default_num_inference_steps)
print("Guidance scale mặc định:", smd.default_guidance_scale)
print("Kích thước sinh:", TARGET_SIZE)
print("Manifest:", RUN_MANIFEST)
print("Số record:", dataset_size)

assert type(smd.scheduler).__name__ == "LCMScheduler", "Notebook phải dùng LCMScheduler từ baseline SemanticDraw SD1.5."


In [ ]:
# 9. Sanity check tùy chọn với một mask phủ toàn ảnh.
if RUN_SANITY_CHECK:
    seed_everything(BASE_SEED)
    sanity_mask = torch.ones(1, 1, TARGET_SIZE[0], TARGET_SIZE[1], dtype=torch.float32, device=device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    tic = time.perf_counter()
    sanity_image = smd(
        prompts=["a studio photo of a teddy bear on a clean table"],
        negative_prompts=[NEGATIVE_PROMPT],
        masks=sanity_mask,
        height=TARGET_SIZE[0],
        width=TARGET_SIZE[1],
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        bootstrap_steps=BOOTSTRAP_STEPS,
        mask_stds=MASK_STD,
        mask_strengths=MASK_STRENGTH,
        preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA,
        do_blend=False,
    )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("[SANITY] Thời gian:", round(time.perf_counter() - tic, 2), "giây")
    print("[SANITY] Thống kê ảnh:", image_stats(sanity_image))
    display(sanity_image.resize((384, 384)))
else:
    print("[INFO] Bỏ qua sanity check.")


In [ ]:
# 10. Sinh ảnh cho manifest đã chọn.
summary_by_index = load_existing_summary()
end_index = dataset_size if MAX_SAMPLES is None else min(dataset_size, START_INDEX + int(MAX_SAMPLES))
print(f"[CHẠY] index {START_INDEX} đến {end_index - 1} trên tổng {dataset_size} record")

stop = False
global_index = 0

for batch_index, batch in enumerate(loader):
    print(f"[BATCH] {batch_index + 1}/{len(loader)} - {len(batch['sample_ids'])} sample")

    for local_index, sample_id in enumerate(batch["sample_ids"]):
        if global_index < START_INDEX:
            global_index += 1
            continue
        if global_index >= end_index:
            stop = True
            break

        payload = make_semanticdraw_payload(batch, local_index)
        sample_id = payload["sample_id"]
        generated_path = generated_path_for(global_index, sample_id)
        overlay_path = overlay_path_for(global_index, sample_id)

        original = batch["images"][local_index].resize((payload["width"], payload["height"]), Image.Resampling.BILINEAR)
        overlay = make_mask_overlay(original, payload["foreground_masks"], payload["category_names"], alpha=0.45)
        overlay.save(overlay_path)

        seed = BASE_SEED + global_index
        elapsed = None
        skipped_existing = False

        if SKIP_EXISTING and generated_path.exists():
            generated = Image.open(generated_path).convert("RGB")
            skipped_existing = True
        else:
            seed_everything(seed)
            masks = payload["all_masks"].to(device=device, dtype=torch.float32)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            tic = time.perf_counter()
            generated = smd(
                prompts=payload["prompts"],
                negative_prompts=payload["negative_prompts"],
                masks=masks,
                height=payload["height"],
                width=payload["width"],
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                mask_stds=MASK_STD,
                mask_strengths=MASK_STRENGTH,
                bootstrap_steps=BOOTSTRAP_STEPS,
                preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA,
                do_blend=False,
            )
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - tic
            generated.save(generated_path)

        row = {
            "index": global_index,
            "batch_index": batch_index,
            "local_index": local_index,
            "sample_id": sample_id,
            "image_id": payload["image_id"],
            "file_name": payload["file_name"],
            "seed": seed,
            "model_family": "sd15",
            "model_id": MODEL_ID,
            "sampler": "LCMScheduler",
            "scheduler": type(smd.scheduler).__name__,
            "num_regions_including_background": len(payload["prompts"]),
            "num_foreground_regions": len(payload["foreground_prompts"]),
            "background_prompt": payload["background_prompt"],
            "foreground_prompts": payload["foreground_prompts"],
            "category_names": payload["category_names"],
            "annotation_ids": payload["annotation_ids"],
            "area_ratios": payload["area_ratios"],
            "elapsed_sec": elapsed,
            "skipped_existing": skipped_existing,
            "generated_stats": image_stats(generated),
            "generated_path": str(generated_path),
            "overlay_path": str(overlay_path),
            "source_manifest_path": str(RUN_MANIFEST),
        }
        summary_by_index[global_index] = row

        if MAX_DISPLAY_RESULTS is None or global_index < START_INDEX + MAX_DISPLAY_RESULTS:
            display_result(payload, original, overlay, generated, elapsed, generated_path)

        with RUN_SUMMARY_PATH.open("w", encoding="utf-8") as f:
            json.dump([summary_by_index[k] for k in sorted(summary_by_index)], f, ensure_ascii=False, indent=2)

        global_index += 1
        del generated, overlay, original
        torch.cuda.empty_cache()

    if stop:
        break

summary = [summary_by_index[k] for k in sorted(summary_by_index)]
with RUN_SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

num_in_range = max(0, end_index - START_INDEX)
display(Markdown(
    f"## Hoàn tất\n"
    f"Đã sinh hoặc dùng lại `{len([r for r in summary if START_INDEX <= int(r['index']) < end_index])}` dòng cho range đang chọn.\n"
    f"Kích thước range: `{num_in_range}`. Tổng số dòng summary hiện có: `{len(summary)}`.\n"
    f"Summary đã lưu tại `{RUN_SUMMARY_PATH}`."
))
pd.DataFrame(summary).tail()


In [ ]:
# 11. Tạo folder export/metric và nén zip để tải về.
if RUN_EXPORT_ZIP:
    EXPORT_MANIFEST_JSONL = RUN_ROOT / "metric_generated_manifest.jsonl"
    EXPORT_MANIFEST_CSV = RUN_ROOT / "metric_generated_manifest.csv"
    EXPORT_SUMMARY_JSON = RUN_ROOT / "export_summary.json"
    ZIP_PATH = BASE_OUTPUT_DIR / f"{RUN_ID}__metric_export.zip"

    manifest_by_sample_id = {}
    with RUN_MANIFEST.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                manifest_by_sample_id[record["sample_id"]] = record

    export_records = []
    for row in summary:
        sample_id = row["sample_id"]
        manifest_record = manifest_by_sample_id.get(sample_id, {})
        generated_path = Path(row["generated_path"])
        if not generated_path.exists():
            continue
        file_name = row.get("file_name", manifest_record.get("file_name"))
        coco_original_path = COCO_ROOT / "val2017" / str(file_name)
        export_records.append({
            "metric_index": int(row["index"]),
            "experiment_id": RUN_ID,
            "sample_id": sample_id,
            "image_id": int(row.get("image_id", manifest_record.get("image_id", -1))),
            "file_name": file_name,
            "generated_image_path": str(generated_path),
            "generated_image_relative_path": str(generated_path.relative_to(RUN_ROOT)),
            "coco_original_path": str(coco_original_path),
            "background_prompt": manifest_record.get("caption", row.get("background_prompt")),
            "foreground_prompts": manifest_record.get("foreground_prompts", row.get("foreground_prompts")),
            "category_names": manifest_record.get("category_names", row.get("category_names")),
            "category_ids": manifest_record.get("category_ids"),
            "annotation_ids": manifest_record.get("annotation_ids", row.get("annotation_ids")),
            "area_ratios": manifest_record.get("area_ratios", row.get("area_ratios")),
            "target_size": manifest_record.get("target_size", list(TARGET_SIZE)),
            "original_size": manifest_record.get("original_size"),
            "model_family": "sd15",
            "sampler": "LCMScheduler",
            "input_protocol": "explicit_background_mask_plus_foreground_masks",
            "seed": row.get("seed"),
            "elapsed_sec": row.get("elapsed_sec"),
            "generation_metadata": row,
        })

    with EXPORT_MANIFEST_JSONL.open("w", encoding="utf-8") as f:
        for record in export_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    csv_fields = [
        "metric_index", "experiment_id", "sample_id", "image_id", "file_name",
        "generated_image_path", "generated_image_relative_path", "coco_original_path",
        "background_prompt", "foreground_prompts", "category_names", "annotation_ids",
        "model_family", "sampler", "seed", "elapsed_sec",
    ]
    with EXPORT_MANIFEST_CSV.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=csv_fields)
        writer.writeheader()
        for record in export_records:
            writer.writerow({
                key: json.dumps(record.get(key), ensure_ascii=False) if isinstance(record.get(key), (list, dict)) else record.get(key)
                for key in csv_fields
            })

    export_summary = {
        "experiment_id": RUN_ID,
        "run_root": str(RUN_ROOT),
        "generated_images_dir": str(GENERATED_IMAGES_DIR),
        "num_export_records": len(export_records),
        "manifest_jsonl": str(EXPORT_MANIFEST_JSONL),
        "manifest_csv": str(EXPORT_MANIFEST_CSV),
        "source_manifest_path": str(RUN_MANIFEST),
        "generation_summary": str(RUN_SUMMARY_PATH),
        "zip_path": str(ZIP_PATH),
    }
    with EXPORT_SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump(export_summary, f, ensure_ascii=False, indent=2)

    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=RUN_ROOT)
    export_summary["zip_size_mb"] = round(ZIP_PATH.stat().st_size / (1024 * 1024), 2)
    with EXPORT_SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump(export_summary, f, ensure_ascii=False, indent=2)

    display(Markdown(
        f"## Export đã sẵn sàng\n"
        f"- thí nghiệm: `{RUN_ID}`\n"
        f"- folder ảnh sinh: `{GENERATED_IMAGES_DIR}`\n"
        f"- manifest dùng cho metric: `{EXPORT_MANIFEST_JSONL}`\n"
        f"- file zip: `{ZIP_PATH}`\n"
        f"- dung lượng zip: `{export_summary['zip_size_mb']} MB`"
    ))
    display(pd.DataFrame(export_records).head())
else:
    print("[INFO] RUN_EXPORT_ZIP=False nên bỏ qua bước tạo zip export.")


## Cell đo metric tùy chọn

Cell bên dưới dùng implementation hiện tại trong `Ours/src/metrics`. Nó hữu ích để so sánh nội bộ, nhưng cần nhớ rằng protocol CLIP hiện tại là region-masked và chưa chắc đã khớp hoàn toàn với paper. Nếu chỉ muốn sinh ảnh và export để đo bằng pipeline khác, hãy giữ `RUN_METRICS = False`.


In [ ]:
# 12. Đo metric tùy chọn.
if RUN_METRICS:
    for var_name in ("smd", "preview_batch", "batch"):
        if var_name in globals():
            del globals()[var_name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    from metrics import MetricEvaluationConfig, run_evaluation, write_metrics_report

    metric_device = "cuda:0" if torch.cuda.is_available() else "cpu"
    metric_config = MetricEvaluationConfig(
        manifest_path=RUN_MANIFEST,
        coco_root=COCO_ROOT,
        generated_dir=GENERATED_IMAGES_DIR,
        generation_summary=RUN_SUMMARY_PATH,
        output_dir=METRICS_OUTPUT_DIR,
        model_family="sd15",
        target_size=TARGET_SIZE,
        metrics=("fid", "is", "clip_fg", "clip_bg", "time"),
        batch_size=METRIC_BATCH_SIZE,
        num_workers=0,
        pin_memory=False,
        device=metric_device,
        clip_batch_size=CLIP_BATCH_SIZE,
        is_splits=10,
    )

    metric_report = run_evaluation(metric_config)
    metrics_json, metrics_csv = write_metrics_report(metric_report, METRICS_OUTPUT_DIR, prefix=f"{RUN_ID}_metrics")
    values = metric_report["metrics"]

    def fmt(value: object, digits: int = 4) -> str:
        if value is None:
            return "-"
        try:
            value = float(value)
            if math.isnan(value):
                return "-"
            return f"{value:.{digits}f}"
        except Exception:
            return str(value)

    metrics_table = pd.DataFrame([
        {"Metric": "FID", "Value": fmt(values.get("fid"))},
        {"Metric": "IS", "Value": fmt(values.get("is_mean"))},
        {"Metric": "IS std", "Value": fmt(values.get("is_std"))},
        {"Metric": "CLIP(fg) x100", "Value": fmt(values.get("clip_fg_x100"))},
        {"Metric": "CLIP(bg) x100", "Value": fmt(values.get("clip_bg_x100"))},
        {"Metric": "Time mean sec", "Value": fmt(values.get("time_mean_sec"))},
        {"Metric": "Total time sec", "Value": fmt(values.get("time_total_sec"))},
    ])

    display(Markdown(
        f"## Đo metric hoàn tất\n"
        f"- đã đánh giá: `{metric_report['num_evaluated']}` / `{metric_report['num_manifest_records']}`\n"
        f"- số ảnh sinh bị thiếu: `{metric_report['num_missing_generated']}`\n"
        f"- metrics JSON: `{metrics_json}`\n"
        f"- metrics CSV: `{metrics_csv}`"
    ))
    display(metrics_table)
else:
    print("[INFO] RUN_METRICS=False nên bỏ qua bước đo metric.")
